# Module 05: Products EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [26]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_products_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

products = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {products.shape}")
products.head(3)

Loaded raw dataset shape: (32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [27]:
audit_df = pd.DataFrame(
    {
        "Data Type": products.dtypes,
        "Missing Values": products.isna().sum(),
        "Missing %": (products.isna().mean() * 100).round(2),
        "Unique Values": products.nunique(),
    }
)
display(audit_df)

,Data Type,Missing Values,Missing %,Unique Values
product_id,object,0,0.00,32951
product_category_name,object,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95


### 3. Missing Value Investigation & Silver Layer Initialization

In [28]:
missing_category_mask = products["product_category_name"].isna()
print(f"Total rows with missing category: {missing_category_mask.sum():,}")

all_descriptive_missing = (
    products["product_category_name"].isna() &
    products["product_name_lenght"].isna() &
    products["product_description_lenght"].isna() &
    products["product_photos_qty"].isna()
)
print(f"Rows with all 4 descriptive fields missing simultaneously: {all_descriptive_missing.sum():,}")

display(products[missing_category_mask].head(3))

Total rows with missing category: 610
Rows with all 4 descriptive fields missing simultaneously: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0


#### Impact Analysis on Order Items
Checking how many order records in `order_items_clean` rely on products with missing metadata to evaluate the risk of data loss if these rows were dropped.

In [29]:
order_items_clean = pd.read_parquet(DATA_DIR / "order_items_clean.parquet")

missing_product_ids = products[products["product_category_name"].isna()]["product_id"]

affected_order_items = order_items_clean[order_items_clean["product_id"].isin(missing_product_ids)]

print(f"Products with missing metadata: {len(missing_product_ids):,}")
print(f"Affected order item records: {len(affected_order_items):,}")

Products with missing metadata: 610
Affected order item records: 1,603


#### Initializing the Silver Layer
Establishing the clean base dataset for the Silver layer by handling missing values in categorical fields to preserve referential integrity.

In [30]:
products_clean = products.copy()

products_clean["product_category_name"] = products_clean["product_category_name"].fillna("unknown")

print(f"Original products count:  {len(products):,}")
print(f"Cleaned products count:   {len(products_clean):,}")
print(f"Filled missing categories: {products['product_category_name'].isna().sum():,}")

Original products count:  32,951
Cleaned products count:   32,951
Filled missing categories: 610


### 4. Text Standardization 

To eliminate discrepancies caused by formatting inconsistencies, we standardize all text fields:
- **String Cleaning:** Strip unwanted whitespaces and convert product categories to lowercase to ensure uniform formatting.
- **Placeholder Normalization:** Filter out empty strings, dashes, and invalid text markers (`n/a`, `null`, `nan`, etc.) and convert them to null values.

In [31]:
products_clean["product_id"] = products_clean["product_id"].astype(str).str.strip()

placeholders = ["", "-", "n/a", "null", "undefined", "nan", "none"]

s = products_clean["product_category_name"].fillna("").astype(str).str.strip().str.lower()
products_clean["product_category_name"] = s.mask(s.isin(placeholders), None)

display(products_clean.head(3))

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


In [32]:
products_clean.isna().sum()

product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

### 5. Data Type Optimization
To transition our dataset from the Bronze layer to the Silver layer, we enforce robust data types:
- **Identifiers & Categories (`string`):** Cast `product_id` and `product_category_name` into explicit string types for reliable relational joins and cross-table integrity.
- **Count & Length Metrics (`Int64`):** Convert discrete integer metrics (name length, description length, and photo quantity) from `float64` to nullable integers to preserve missing values cleanly without coercion[cite: 10].
- **Physical Measurements (`float64`):** Retain standard floating-point representation for weight and dimensional attributes to maintain statistical continuity for downstream aggregations[cite: 10].

In [33]:
products_clean["product_id"] = products_clean["product_id"].astype("string")
products_clean["product_category_name"] = products_clean["product_category_name"].astype("string")

integer_metrics = [
    "product_name_lenght", 
    "product_description_lenght", 
    "product_photos_qty"
]
for col in integer_metrics:
    products_clean[col] = products_clean[col].astype("Int64")

display(products_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
product_id,string[python]
product_category_name,string[python]
product_name_lenght,Int64
product_description_lenght,Int64
product_photos_qty,Int64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64


### 6. Data Integrity Checks & Anomaly Detection

Ensuring logical consistency, validating business rules, and inspecting value distributions to isolate structural anomalies and outliers:

In [34]:
audit_checks = {
    "Duplicate product IDs": (
        products_clean["product_id"].duplicated().sum()
    ),

    "Non-positive product weight": (
        products_clean["product_weight_g"] <= 0
    ),
    "Non-positive product dimensions": (
        (products_clean["product_length_cm"] <= 0) | 
        (products_clean["product_height_cm"] <= 0) | 
        (products_clean["product_width_cm"] <= 0)
    ),
    "Non-positive text lengths": (
        (products_clean["product_name_lenght"] <= 0) | 
        (products_clean["product_description_lenght"] <= 0)
    ),
    "Negative photo quantity": (
        products_clean["product_photos_qty"] < 0
    ),
    
    "Weight > 0 but zero dimensions": (
        (products_clean["product_weight_g"] > 0) & 
        (
            (products_clean["product_length_cm"] == 0) | 
            (products_clean["product_height_cm"] == 0) | 
            (products_clean["product_width_cm"] == 0)
        )
    ),

    "Unusually high photo quantity (>20)": (
        products_clean["product_photos_qty"] > 20
    )
}

for label, condition in audit_checks.items():
    count = condition.sum() if hasattr(condition, "sum") else condition
    print(f"{label}: {count:,}")

Duplicate product IDs: 0
Non-positive product weight: 4
Non-positive product dimensions: 0
Non-positive text lengths: 0
Negative photo quantity: 0
Weight > 0 but zero dimensions: 0
Unusually high photo quantity (>20): 0


#### Inspecting Weight Anomalies
Isolating the records with non-positive or zero weight to inspect their attributes and determine the appropriate cleaning strategy.

In [35]:
problematic_weights = products_clean[products_clean["product_weight_g"] <= 0]
display(problematic_weights)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51,529,1,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48,528,1,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53,528,1,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53,528,1,0.0,30.0,25.0,30.0


#### Handling Weight Anomalies
Replacing zero-weight anomalies with `NaN` to maintain statistical integrity across the dataset.

In [36]:
anomalous_weight_mask = products_clean["product_weight_g"] <= 0
print(f"Anomalous weight records found: {anomalous_weight_mask.sum()}")

products_clean.loc[anomalous_weight_mask, "product_weight_g"] = None

print(f"Remaining zero/negative weights: {(products_clean['product_weight_g'] <= 0).sum():,}")
print(f"Missing weights count now: {products_clean['product_weight_g'].isna().sum():,}")

Anomalous weight records found: 4
Remaining zero/negative weights: 0
Missing weights count now: 6


### 7. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [37]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {products.shape}")
print(f"Cleaned shape (Silver): {products_clean.shape}")
print(f"Total rows dropped: {len(products) - len(products_clean):,}")

dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': products.dtypes,
    'Silver (Clean)': products_clean.dtypes
})

print("\nData Types Transformation:")
display(dtypes_comparison)

print("\nFinal Missing Values Audit (Silver Layer):")
final_audit = pd.DataFrame({
    "Missing Values": products_clean.isna().sum(),
    "Missing %": (products_clean.isna().mean() * 100).round(2)
})

final_audit = final_audit[final_audit["Missing Values"] > 0]
display(final_audit)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (32951, 9)
Cleaned shape (Silver): (32951, 9)
Total rows dropped: 0

Data Types Transformation:


,Bronze (Raw),Silver (Clean)
product_id,object,string[python]
product_category_name,object,string[python]
product_name_lenght,float64,Int64
product_description_lenght,float64,Int64
product_photos_qty,float64,Int64
product_weight_g,float64,float64
product_length_cm,float64,float64
product_height_cm,float64,float64
product_width_cm,float64,float64



Final Missing Values Audit (Silver Layer):


,Missing Values,Missing %
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,6,0.02
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


## 8. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks tailored to product catalog integrity and physical dimensions:

* **Primary Key Integrity:** Ensure `product_id` is entirely unique and free of missing values.
* **Physical Measurement Constraints:** Validate that product weight and dimensions (`length`, `height`, `width`) contain no non-positive anomalies.
* **Text & Metric Validity:** Confirm that text lengths and photo quantities are within expected non-negative boundaries.
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [38]:
assert products_clean['product_id'].is_unique, "Assertion failed: Duplicate product_id found"
assert products_clean['product_id'].notna().all(), "Assertion failed: NULL values found in product_id"

assert (products_clean['product_weight_g'].dropna() > 0).all(), "Assertion failed: Non-positive product weight found"
assert (products_clean['product_length_cm'].dropna() > 0).all(), "Assertion failed: Non-positive product length found"
assert (products_clean['product_height_cm'].dropna() > 0).all(), "Assertion failed: Non-positive product height found"
assert (products_clean['product_width_cm'].dropna() > 0).all(), "Assertion failed: Non-positive product width found"

assert (products_clean['product_name_lenght'].dropna() > 0).all(), "Assertion failed: Non-positive product name length found"
assert (products_clean['product_description_lenght'].dropna() > 0).all(), "Assertion failed: Non-positive product description length found"
assert (products_clean['product_photos_qty'].dropna() >= 0).all(), "Assertion failed: Negative product photos quantity found"

assert len(products_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
products_clean.to_parquet("../data/products_clean.parquet", index=False)
print("Products dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.
Products dataset successfully saved to Parquet.
